# Notebook 6: Experiment 2 — Cross-Stock Prediction (70/30)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on one stock's daily data, predict on another stock's daily test data.  
**Train/Test Split:** 70/30 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501 BBCA ATH)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.7
RATIO_LABEL = '70_30'
EXP_LABEL = f'Exp2_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 2 - Cross-Stock Prediction (70/30)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 2 - Cross-Stock Prediction (70/30)


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


In [3]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


## Run All Cross-Stock Experiments

### Methodology: Zero-Shot Transfer Learning
Instead of retraining from scratch, we load pre-trained models from **Experiment 1** (with same-stock data) and directly apply them to predict different stocks' test data. This tests how well models generalize across stocks without any domain adaptation.

**Expected result:** Performance will likely be worse than same-stock predictions due to different price ranges, volatility, and patterns across stocks. However, this shows raw transfer capability.

In [4]:
# ============================================================
# EXPERIMENT 2: Cross-stock prediction (using Exp1 pre-trained models)
# Load models trained on Stock A, test on Stock B (zero-shot transfer)
# ============================================================
from tensorflow.keras.models import load_model

all_results = []
all_predictions = {}  # {(train_stock, test_stock): {model_type: (y_true, y_pred, dates)}}

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue  # Skip same-stock (covered in Exp 1)
        
        pair_key = (train_stock, test_stock)
        print(f"\n{'#'*60}")
        print(f"# TRAIN: {train_stock} -> TEST: {test_stock}")
        print(f"# Using pre-trained Exp1 models (zero-shot transfer)")
        print(f"{'#'*60}")
        
        # Prepare cross-stock data
        X_train, y_train, X_test, y_test, test_dates = prepare_cross_stock_data(
            daily_data[train_stock], daily_data[test_stock],
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        all_predictions[pair_key] = {}
        
        for model_type in MODEL_TYPES:
            # Build Exp1 model filename
            exp1_model_path = f'models/Exp1_70_30/Exp1_70_30_{train_stock}_{model_type}_best.keras'
            
            if not os.path.exists(exp1_model_path):
                print(f"  ⚠️  Model not found: {exp1_model_path}")
                continue
            
            print(f"\n  Loading {model_type} from: {exp1_model_path}")
            
            # Load pre-trained model
            model = load_model(exp1_model_path)
            
            # Make predictions on test data (NO training)
            y_pred_scaled = model.predict(X_test, verbose=0).flatten()
            
            # Inverse scale to original values
            y_true_inv = proportion_inverse_scale(y_test)
            y_pred_inv = proportion_inverse_scale(y_pred_scaled)
            
            # Evaluate
            metrics = evaluate_predictions(y_true_inv, y_pred_inv)
            
            result = {
                'Train_Stock': train_stock,
                'Test_Stock': test_stock,
                'Model': model_type,
                **metrics
            }
            all_results.append(result)
            all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
            
            print(f"    RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}, R²: {metrics['R2']:.6f}")
            
            # Plot prediction
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'Train_{train_stock}_Test_{test_stock}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 2 (70/30) cross-stock prediction complete!")


############################################################
# TRAIN: TLKM -> TEST: BBCA
# Using pre-trained Exp1 models (zero-shot transfer)
############################################################
  X_train: (3669, 1, 1), X_test: (1574, 1, 1)

  Loading BiLSTM from: models/Exp1_70_30/Exp1_70_30_TLKM_BiLSTM_best.keras
    RMSE: 242.7185, MAE: 199.0013, R²: 0.976414
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_BBCA_BiLSTM_prediction.png

  Loading BiGRU from: models/Exp1_70_30/Exp1_70_30_TLKM_BiGRU_best.keras
    RMSE: 257.6404, MAE: 215.6482, R²: 0.973425
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_BBCA_BiGRU_prediction.png

  Loading LSTM from: models/Exp1_70_30/Exp1_70_30_TLKM_LSTM_best.keras
    RMSE: 308.4701, MAE: 257.0866, R²: 0.961905
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_BBCA_LSTM_prediction.png

  Loading GRU from: models/Exp1_70_30/Exp1_70_30_TLKM_GRU_best.keras
    RMSE: 162.8648, MAE: 127.2353, R²: 0.989381
  

## Results Summary

In [5]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 2 - Cross-Stock Prediction (70/30)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 2 - Cross-Stock Prediction (70/30)
Train_Stock Test_Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2
       TLKM       BBCA BiLSTM 58912.2831 242.7185 199.0013    2.5709 0.976414
       TLKM       BBCA  BiGRU 66378.5602 257.6404 215.6482    2.8009 0.973425
       TLKM       BBCA   LSTM 95153.8166 308.4701 257.0866    3.2813 0.961905
       TLKM       BBCA    GRU 26524.9557 162.8648 127.2353    1.7020 0.989381
       TLKM       ASII BiLSTM  9221.2483  96.0273  72.3699    1.6447 0.982671
       TLKM       ASII  BiGRU 10411.5169 102.0368  77.9963    1.7596 0.980434
       TLKM       ASII   LSTM 10367.8679 101.8227  77.3215    1.7407 0.980516
       TLKM       ASII    GRU  8100.8398  90.0047  67.2229    1.5464 0.984776
       TLKM       UNVR BiLSTM 11170.7155 105.6916  72.4858    1.8712 0.996246
       TLKM       UNVR  BiGRU 12719.6210 112.7813  78.0798    1.9913 0.995725
       TLKM       UNVR   LSTM 13775.5971 117.3695  80.5334    1.9988 0.995371
       TLKM    

## Visualizations

In [6]:
# ============================================================
# HEATMAPS PER MODEL
# ============================================================
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_heatmap(
        results_df, metric, EXP_LABEL,
        row_col='Train_Stock', col_col='Test_Stock',
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All heatmaps saved!")


  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiLSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiGRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_LSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_GRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiLSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiGRU_MAE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_LSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_GRU_MAE_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiLSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiGRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_LSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_GRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiLSTM_R2_heatmap.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_BiGRU_R2_heatmap.png
  Figure saved: figures/Exp2_70

In [7]:
# ============================================================
# COMPARISON: All models for each train->test pair
# ============================================================
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        preds = {mt: all_predictions[pair_key][mt][1] for mt in MODEL_TYPES if mt in all_predictions[pair_key]}
        
        plot_all_models_comparison(
            dates, y_true, preds,
            f'Train_{train_stock}_Test_{test_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("All comparison plots saved!")


  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_ASII_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_TLKM_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_BBCA_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_BBCA_Test_ASII_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_BBCA_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_ASII_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_ASII_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_ASII_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_UNVR_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_UNVR_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_70_30/Exp2_70_30_Train_UNVR_Test_ASII_all_models.png
All comparison p

In [8]:
# ============================================================
# SUMMARY: BEST MODEL PER CROSS-STOCK PAIR
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER PAIR (by RMSE)")
print("="*70)
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_data = results_df[
            (results_df['Train_Stock'] == train_stock) &
            (results_df['Test_Stock'] == test_stock)
        ]
        if pair_data.empty:
            continue
        best_idx = pair_data['RMSE'].idxmin()
        best = pair_data.loc[best_idx]
        print(f"  {train_stock} -> {test_stock}: {best['Model']} "
              f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST MODEL PER PAIR (by RMSE)
  TLKM -> BBCA: GRU (RMSE=162.8648, R²=0.989381)
  TLKM -> ASII: GRU (RMSE=90.0047, R²=0.984776)
  TLKM -> UNVR: GRU (RMSE=93.7107, R²=0.997049)
  BBCA -> TLKM: BiGRU (RMSE=54.5462, R²=0.985874)
  BBCA -> ASII: BiGRU (RMSE=83.2079, R²=0.986989)
  BBCA -> UNVR: BiGRU (RMSE=85.9952, R²=0.997515)
  ASII -> TLKM: GRU (RMSE=60.1222, R²=0.982838)
  ASII -> BBCA: GRU (RMSE=145.5719, R²=0.991516)
  ASII -> UNVR: GRU (RMSE=90.4188, R²=0.997253)
  UNVR -> TLKM: LSTM (RMSE=59.3669, R²=0.983267)
  UNVR -> BBCA: BiGRU (RMSE=115.9619, R²=0.994616)
  UNVR -> ASII: LSTM (RMSE=86.9563, R²=0.985790)


In [9]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS - CROSS-STOCK
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Cross-Stock Results Dashboard
print("1. Generating Cross-Stock Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp2(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Transfer Learning Matrix Heatmaps
print("2. Generating Transfer Learning Matrices...")
for metric in ['RMSE', 'MAE', 'R2']:
    try:
        create_interactive_transfer_matrix(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"   ✓ {metric} transfer matrices generated")
    except Exception as e:
        print(f"   ⚠ Skipping {metric}: {str(e)}")

print()

# 3. Metrics Comparison Chart
print("3. Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("4. Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html4}")
fig4.show()

print("\n✓ All interactive results visualizations generated successfully!")

Generating interactive results visualizations...

1. Generating Cross-Stock Results Dashboard...
   ✓ Saved: figures/Exp2_70_30/Exp2_70_30_crossstock_dashboard.html



2. Generating Transfer Learning Matrices...
   ✓ RMSE transfer matrices generated
   ✓ MAE transfer matrices generated
   ✓ R2 transfer matrices generated

3. Generating Metrics Comparison Chart...
   ✓ Saved: figures/Exp2_70_30/Exp2_70_30_metrics_comparison.html



4. Generating Model Radar Chart...
   ✓ Saved: figures/Exp2_70_30/Exp2_70_30_model_radar.html



✓ All interactive results visualizations generated successfully!


## Interactive Results Visualizations